# 03 · The 🤗 `datasets` library

After this notebook you can load data from Python objects, pandas, JSON and CSV, clean and tokenise it with `map`/`filter`, split it without leaking or skewing labels, feed it to PyTorch, save it and stream it. You will also build the two dataset shapes fine-tuning needs: an **SFT chat dataset** and a **preference dataset**.

**Time:** ~25 min · **Runs:** offline on CPU in < 1 min (all data is inline or written to a temp folder) · **Needs:** [pandas essentials](../04_numpy_pandas_pytorch/02_pandas_essentials.ipynb), [02 · Tokenizers](02_tokenizers_models_and_generate.ipynb)

## Why this matters in interviews

- Every fine-tuning question ends up asking "what does the data look like?". You should be able to write the `messages` format and the `prompt/chosen/rejected` format from memory, and explain why each one exists.
- "The corpus doesn't fit in RAM" has a specific answer here: Arrow memory-mapping and streaming mode.
- Data bugs cause most failed fine-tunes: duplicates across train and test, skewed splits, silent truncation. The tools in this notebook are how you catch them.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `dt01` | You need to process a 5 GB CSV on a server with 2 GB of RAM. What do you do? |
| `dt03` | "Streaming" means at least four different things. Which do you mean? |
| `tn13` | How much data do you need to fine-tune, and what does good data look like? |
| `tn25` | How would you build a fine-tuning dataset from production traffic? |
| `tn37` | What is the difference between SFT and preference tuning, in terms of what each can teach? |
| `dt22` | How do you sample data for evaluation without biasing the result? |

In [ ]:
import os, json, time, tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import datasets
from datasets import Dataset, DatasetDict, ClassLabel, Value, Features, load_dataset, load_from_disk
import transformers
from transformers import AutoTokenizer

datasets.disable_progress_bars()
transformers.utils.logging.set_verbosity_error()
rng = np.random.default_rng(0)
WORK = Path(tempfile.mkdtemp())          # every file this notebook writes goes here, never into the repo
print("datasets", datasets.__version__, "| scratch folder:", WORK)

## 1. What a `Dataset` is

**In plain English:** a `Dataset` is a table, like a DataFrame, but stored in the **Apache Arrow** columnar format. The table can live on disk and be *memory-mapped*: the operating system pages rows in only when you read them. That lets you work with a 100 GB corpus on a 16 GB laptop, and it is why `datasets` exists at all.

You can build one from Python objects in three ways:

In [ ]:
from_dict = Dataset.from_dict({"text": ["refund please", "app crashes"], "label": [0, 1]})
from_list = Dataset.from_list([{"text": "refund please", "label": 0}, {"text": "app crashes", "label": 1}])
from_pandas = Dataset.from_pandas(pd.DataFrame({"text": ["refund please", "app crashes"], "label": [0, 1]}))

print(from_dict)
print("features:", from_dict.features)
assert from_dict[:] == from_list[:] == from_pandas[:]    # same content, three constructors
print("row 0   :", from_dict[0])        # a dict for one row
print("rows 0-1:", from_dict[:2])       # a dict of LISTS for a slice (column-oriented!)
print("column  :", list(from_dict["text"]))

A gotcha: `ds[0]` gives a dict for one row, but `ds[:2]` gives **a dict of lists**, because Arrow stores data by column. Batched functions receive the same shape (section 3).

Now a realistic toy corpus: 240 support tickets in three classes, plus the mess real data has (exact duplicates, empty rows, one very long rant):

In [ ]:
templates = {
    "billing":   ["I was charged twice for {x}", "Refund my {x} payment", "Why is my {x} invoice so high",
                  "Cancel the extra {x} charge"],
    "technical": ["The {x} page crashes on load", "Error 500 when I open {x}", "{x} sync is not working",
                  "The {x} export button does nothing"],
    "account":   ["Reset my password for {x}", "Change the email on my {x} profile", "Delete my {x} account",
                  "I am locked out of {x}"],
}
products = ["Pro", "Team", "mobile", "desktop", "Enterprise", "starter"]
rows = []
for label, temps in templates.items():
    n = {"billing": 100, "technical": 90, "account": 50}[label]      # deliberately imbalanced
    for _ in range(n):
        rows.append({"text": rng.choice(temps).format(x=rng.choice(products)), "label": label,
                     "channel": str(rng.choice(["email", "chat", "phone"]))})
rows += [{"text": "", "label": "technical", "channel": "chat"},
         {"text": "   ", "label": "billing", "channel": "email"},
         {"text": "This is unacceptable " * 60, "label": "billing", "channel": "email"}]
pd.DataFrame(rows).to_json(WORK / "tickets.jsonl", orient="records", lines=True)
pd.DataFrame(rows).to_csv(WORK / "tickets.csv", index=False)
print(f"wrote {len(rows)} rows to tickets.jsonl and tickets.csv")

## 2. Loading files with `load_dataset`

`load_dataset` takes either a Hub repo id (`load_dataset("stanfordnlp/imdb")`) or a **builder name plus local files**: `"json"`, `"csv"`, `"parquet"`, `"text"`. It converts the files to Arrow once, caches the result under `~/.cache/huggingface/datasets`, and returns a `DatasetDict` of splits.

In [ ]:
raw = load_dataset("json", data_files=str(WORK / "tickets.jsonl"))
print(raw)                                         # a DatasetDict with a single "train" split
csv_split = load_dataset("csv", data_files={"train": str(WORK / "tickets.csv")}, split="train")
print("csv rows:", csv_split.num_rows, "| features:", csv_split.features)
assert raw["train"].num_rows == csv_split.num_rows == len(rows)

ds = raw["train"]
print("\nbacked by:", type(ds.data).__name__, "->", Path(ds.cache_files[0]["filename"]).name)

`MemoryMappedTable` means the rows live in an Arrow file in the cache, and Python only holds a handle to it. A dataset built with `from_dict` is an `InMemoryTable` instead, until you save it (section 6).

### Features: the schema

Every column has a type. For classification, turn string labels into a `ClassLabel`. It stores integers and keeps the names, which gives you `id2label` for the model config.

In [ ]:
label_names = ["billing", "technical", "account"]
ds = ds.cast_column("label", ClassLabel(names=label_names))
print(ds.features)
print("row 0:", ds[0], "->", ds.features["label"].int2str(ds[0]["label"]))
id2label = dict(enumerate(ds.features["label"].names))
print("id2label for the model config:", id2label)
assert ds.features["label"].str2int("account") == 2

## 3. Cleaning: `filter`, `map`, and friends

| Method | Does | Note |
|---|---|---|
| `filter(fn)` | keep the rows where `fn(row)` is True | `batched=True` for speed |
| `map(fn, batched=True)` | add or replace columns | the workhorse: cleaning, tokenising, labelling |
| `select(indices)` | pick rows by index | cheap: it only builds an index mapping |
| `shuffle(seed=...)` | permute the rows | always pass a seed |
| `train_test_split(test_size, seed, stratify_by_column)` | two splits | stratify for imbalanced labels |
| `rename_column`, `remove_columns`, `select_columns` | reshape the schema | Trainer drops unknown columns anyway |
| `unique(col)`, `sort(col)`, `to_pandas()` | inspect | `to_pandas` loads everything into RAM |

In [ ]:
before = ds.num_rows
ds = ds.filter(lambda r: len(r["text"].strip()) > 0)                  # drop empty / whitespace-only rows
print(f"filter empties: {before} -> {ds.num_rows}")

seen = set()
def first_time(row):                     # exact-duplicate removal: fine for small data, single process
    key = row["text"].strip().lower()
    if key in seen:
        return False
    seen.add(key)
    return True
before = ds.num_rows
ds = ds.filter(first_time)
print(f"drop exact duplicates: {before} -> {ds.num_rows}  (templates repeat a lot)")
assert len(set(t.strip().lower() for t in ds["text"])) == ds.num_rows

Duplicates matter more than they seem. If the same ticket lands in both train and test, the test score rewards memorisation. Always deduplicate **before** splitting. At scale you would hash normalised text, or use MinHash for near-duplicates (`tn25`).

Now tokenise with `map(batched=True)`. The function receives a batch as a dict of lists and returns new columns. Batching matters because fast tokenizers process a whole list in parallel Rust code:

In [ ]:
tok = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=32)

big = ds.select([i % ds.num_rows for i in range(1000)])          # repeat rows to get something to time
timings = {}
for name, kwargs in {"one row at a time": dict(batched=False), "batched (1000 rows)": dict(batched=True)}.items():
    t0 = time.perf_counter()
    big.map(tokenize, **kwargs, load_from_cache_file=False)
    timings[name] = time.perf_counter() - t0
    print(f"{name:20s}: {timings[name]:.2f}s for {big.num_rows} rows")

tokenized = ds.map(tokenize, batched=True)
print("\nnew columns:", tokenized.column_names)
print("first row input_ids:", tokenized[0]["input_ids"])

Batched mapping is usually several times faster, and the gap grows with dataset size. Two more `map` facts interviewers like: results are **cached by fingerprint** (running the same `map` twice on the same data is instant), and `num_proc=4` runs it in parallel processes.

Truncation is silent, so check what you cut. The rant row is far longer than 32 tokens:

In [ ]:
lengths = [len(x) for x in ds.map(lambda b: tok(b["text"]), batched=True)["input_ids"]]
print(f"longest ticket: {max(lengths)} tokens -> truncated to 32; "
      f"{sum(l > 32 for l in lengths)} of {len(lengths)} rows lose text")
assert max(lengths) > 32

## 4. Splitting without skew

Plain random splits of **imbalanced, small** data give test sets with the wrong class mix. `stratify_by_column` keeps the proportions the same in both splits (`dt22`). It needs a `ClassLabel` column, which is one more reason to cast labels.

In [ ]:
def class_share(d):
    counts = np.bincount(d["label"], minlength=3)
    return (counts / counts.sum()).round(2)

plain = tokenized.train_test_split(test_size=0.2, seed=3)
strat = tokenized.train_test_split(test_size=0.2, seed=3, stratify_by_column="label")
print("full data  :", class_share(tokenized))
print("plain test :", class_share(plain["test"]))
print("strat test :", class_share(strat["test"]))
assert np.abs(class_share(strat["test"]) - class_share(tokenized)).max() <= 0.03

splits = DatasetDict(train=strat["train"], test=strat["test"])
splits = splits.remove_columns(["channel"]).rename_column("label", "labels")   # the name HF models expect
print(splits)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
x = np.arange(3); w = 0.27
for i, (name, d) in enumerate({"full": tokenized, "plain split (test)": plain["test"],
                               "stratified split (test)": strat["test"]}.items()):
    ax.bar(x + (i - 1) * w, class_share(d), w, label=name)
ax.set_xticks(x, label_names); ax.set_ylabel("share of rows")
ax.set_title("Stratified splitting keeps the class mix of the test set"); ax.legend(fontsize=8)
plt.show()

## 5. Into PyTorch: `set_format` + `DataLoader`

`set_format("torch", columns=[...])` makes indexing return tensors. The padding is done **per batch** by a collator (`DataCollatorWithPadding`), so each batch is padded only to its own longest row. That "dynamic padding" is much cheaper than padding everything to `max_length`.

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

train = splits["train"].remove_columns(["text"])
collate = DataCollatorWithPadding(tok)                 # pads input_ids/attention_mask, stacks labels
loader = DataLoader(train, batch_size=8, shuffle=True, collate_fn=collate,
                    generator=torch.Generator().manual_seed(0))
first = next(iter(loader))
print("one batch:", ", ".join(f"{k} {tuple(v.shape)}" for k, v in first.items()))
widths = [b["input_ids"].shape[1] for b in loader]
print("padded width of every batch in one epoch:", widths)
assert len(set(widths)) > 1          # each batch is padded only to its own longest row

splits.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
print("\nwith set_format('torch'):", {k: type(v).__name__ for k, v in splits["test"][0].items()})
splits.reset_format()

The widths differ from batch to batch because each batch is padded only to its own longest row. That is dynamic padding. Padding every row to `max_length=32` would roughly triple the compute spent on these short tickets. The `Trainer` in [notebook 05](05_trainer_api_fine_tune_small_model.ipynb) does exactly this for you when you pass the collator.

## 6. Saving, memory-mapping and streaming

| Tool | Use it when |
|---|---|
| `save_to_disk` / `load_from_disk` | caching your processed dataset between runs (Arrow files, memory-mapped on load) |
| `to_parquet` / `load_dataset("parquet", ...)` | sharing: compressed, columnar, readable by pandas, DuckDB and Spark |
| `push_to_hub("you/name", private=True)` | sharing on the Hub (needs `HF_TOKEN`; not run here) |
| `load_dataset(..., streaming=True)` | the data is bigger than your disk or RAM, or you only need the first N rows |

In [ ]:
n = 500_000
numbers = Dataset.from_dict({"x": np.arange(n, dtype=np.float32), "y": rng.normal(size=n).astype(np.float32)})
path = WORK / "numbers"
numbers.save_to_disk(str(path))

t0 = time.perf_counter()
reloaded = load_from_disk(str(path))
print(f"load_from_disk: {(time.perf_counter() - t0) * 1000:.0f} ms for {reloaded.num_rows:,} rows "
      f"({sum(f.stat().st_size for f in path.glob('*.arrow')) / 1e6:.0f} MB on disk)")
print("in memory before saving:", type(numbers.data).__name__, "| after load:", type(reloaded.data).__name__)
assert type(reloaded.data).__name__ == "MemoryMappedTable"
print("row 123,456:", reloaded[123_456])        # random access without reading the rest

Loading half a million rows is fast because nothing is read yet. The OS maps the file and pages in only the rows you touch. This is the `dt01` answer in 🤗 terms: the file size is not the constraint, the largest object you hold in memory is.

**Streaming** goes one step further: `streaming=True` returns an `IterableDataset` that reads records lazily, from local files or directly from the Hub, without downloading or converting everything first. You lose random access and `len()`; you get constant memory. `map`, `filter`, `take` and `shuffle(buffer_size=...)` are applied lazily as you iterate. (`dt03`: this is *data* streaming, not token streaming from an LLM API.)

In [ ]:
stream = load_dataset("json", data_files=str(WORK / "tickets.jsonl"), split="train", streaming=True)
print(type(stream).__name__)

pipeline = (stream
            .filter(lambda r: r["text"].strip() != "")
            .map(lambda r: {"n_words": len(r["text"].split())})
            .shuffle(seed=0, buffer_size=50))          # approximate shuffle: a 50-row buffer
for r in pipeline.take(3):
    print(r)

# a full pass with O(1) memory: count words per label without ever holding the dataset
totals = {}
for r in stream.filter(lambda r: r["text"].strip() != ""):
    totals[r["label"]] = totals.get(r["label"], 0) + len(r["text"].split())
print("words per label:", totals)

## 7. Fine-tuning dataset formats

The two formats you should be able to write from memory. TRL's `SFTTrainer` and `DPOTrainer` accept these column names directly.

### SFT: a `messages` column

Each row is one conversation in the **same role/content format** that `apply_chat_template` renders. SFT teaches the model *what a good answer looks like*: format, tone, domain vocabulary, tool-call syntax (`tn37`).

In [ ]:
sft_rows = [
    {"messages": [
        {"role": "system", "content": "You are a support agent for Acme. Be brief."},
        {"role": "user", "content": q},
        {"role": "assistant", "content": a}]}
    for q, a in [
        ("I was charged twice this month.", "Sorry about that! I've refunded the duplicate charge; it arrives in 3-5 days."),
        ("The app crashes when I open it.", "Please update to the latest version. If it still crashes, send us the error code."),
        ("How do I reset my password?", "Click 'Forgot password' on the login page and follow the email link."),
        ("Can I pay by bank transfer?", "Yes, on annual plans. Choose 'Invoice' at checkout."),
        ("Delete my account.", "I can do that. Please confirm from your registered email first; deletion is permanent."),
        ("Why is my invoice higher than last month?", "Your team added two seats on the 12th; the invoice includes a pro-rated charge for them."),
    ]]
sft = Dataset.from_list(sft_rows)
print(sft)
print(json.dumps(sft[0]["messages"], indent=1)[:260], "...")

smol = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")
sft = sft.map(lambda r: {"n_tokens": len(smol.apply_chat_template(r["messages"], tokenize=True)["input_ids"])})
print("\ntokens per conversation (chat template included):", list(sft["n_tokens"]))
assert all(n > 30 for n in sft["n_tokens"])

### Preference data: `prompt` / `chosen` / `rejected`

Preference tuning (DPO and its relatives) learns from **pairs**: for the same prompt, which of two answers is better. It can teach things SFT struggles with, such as "don't over-promise", "ask before a destructive action", or "be concise", because the model sees the bad answer as well as the good one ([DPO notebook](../11_finetuning/06_dpo_preference_tuning.ipynb)).

In [ ]:
pref = Dataset.from_list([
    {"prompt": "Delete my account.",
     "chosen": "I can do that. Please confirm from your registered email first; deletion is permanent.",
     "rejected": "Done! Your account and all data are deleted."},
    {"prompt": "Will my refund arrive today?",
     "chosen": "Refunds usually take 3-5 business days; I can't promise today, sorry.",
     "rejected": "Yes, definitely today!"},
    {"prompt": "The app crashes when I open it.",
     "chosen": "Please update to the latest version. If it still crashes, send us the error code.",
     "rejected": "Works on my machine."},
])
print(pref)
for r in pref.select(range(1)):
    print(json.dumps(r, indent=1))
assert set(pref.column_names) == {"prompt", "chosen", "rejected"}

TRL also accepts a **conversational** form of the same thing: `prompt` is a list of messages, and `chosen`/`rejected` are one-message lists with the assistant reply. Where the data comes from (`tn25`): SFT rows from resolved tickets or approved answers; preference pairs from thumbs-up/down, from a human edit of a model draft (the edit is "chosen", the draft "rejected"), or from A/B choices. Scrub PII first, deduplicate, and hold out an eval set **before** you train.

### How long are the examples? Pick `max_length` from data

In [ ]:
chat_lengths = ds.map(lambda b: {"n": [len(smol.apply_chat_template(
    [{"role": "user", "content": t}, {"role": "assistant", "content": "Thanks, we are on it."}])["input_ids"])
    for t in b["text"]]}, batched=True)["n"]
chat_lengths = np.array(chat_lengths)
p95 = np.percentile(chat_lengths, 95)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(chat_lengths, bins=40, color="#4c6ef5")
ax.axvline(p95, color="#e03131", ls="--", label=f"95th percentile = {p95:.0f} tokens")
ax.set_xlabel("tokens per example (chat template included)"); ax.set_ylabel("rows")
ax.set_title("Token lengths decide max_length (and memory)"); ax.legend()
plt.show()
print(f"median {np.median(chat_lengths):.0f}, p95 {p95:.0f}, max {chat_lengths.max()} tokens")
assert chat_lengths.max() > 4 * np.median(chat_lengths)       # one outlier dwarfs the rest

The chat template alone adds about 30 tokens to every row (SmolLM2 injects its default system prompt). Most rows are short, and one outlier is several times longer. Setting `max_length` to the maximum would waste memory on padding for every batch; setting it near the 95th percentile and reviewing what gets truncated is the usual compromise. For SFT you also usually **drop** examples that would be truncated, because a cut-off assistant answer teaches the model to stop mid-sentence.

## Try it yourself

**1. Channel filter + `to_pandas`.** Keep only chat-channel tickets, then use `to_pandas()` and `value_counts()` to count labels.

In [ ]:
# Solution — try it yourself first, then run this
chat_only = ds.filter(lambda r: r["channel"] == "chat")
counts = chat_only.to_pandas()["label"].map(dict(enumerate(label_names))).value_counts()
print(counts)
assert counts.sum() == chat_only.num_rows

**2. Batched filter.** Rewrite "drop rows with fewer than 5 words" as a **batched** filter. The function gets a dict of lists and must return a list of booleans.

In [ ]:
# Solution — try it yourself first, then run this
long_enough = ds.filter(lambda b: [len(t.split()) >= 5 for t in b["text"]], batched=True)
row_by_row = ds.filter(lambda r: len(r["text"].split()) >= 5)
print(f"{ds.num_rows} -> {long_enough.num_rows} rows")
assert list(long_enough["text"]) == list(row_by_row["text"])

**3. SFT rows from the tickets.** Turn each deduplicated ticket into an SFT conversation with a canned reply per label (`"billing" → "Our billing team will review this today."`, and so on), then check that every row renders with the chat template.

In [ ]:
# Solution — try it yourself first, then run this
replies = {0: "Our billing team will review this today.", 1: "Our engineers are looking into it.",
           2: "We have sent you a secure link to manage your account."}
to_sft = ds.map(lambda r: {"messages": [{"role": "user", "content": r["text"]},
                                        {"role": "assistant", "content": replies[r["label"]]}]},
                remove_columns=ds.column_names)
print(to_sft, "\n", to_sft[0]["messages"])
assert all(smol.apply_chat_template(m, tokenize=False).count("<|im_start|>assistant") == 1
           for m in to_sft["messages"])

## Say it in an interview

- **What `datasets` is:** "Arrow-backed tables that are memory-mapped from disk, with cached, fingerprinted `map`/`filter` transforms and a streaming mode. So dataset size is limited by disk, not RAM, and a 100 GB corpus is workable on a laptop."
- **Pipeline order:** load → cast labels to `ClassLabel` → filter empties → **deduplicate before splitting** → stratified `train_test_split(seed=...)` → `map(tokenize, batched=True)` → dynamic padding with a collator → `set_format("torch")` or hand it to the Trainer.
- **Formats:** SFT = a `messages` column in role/content form, rendered with the model's chat template; preference = `prompt` / `chosen` / `rejected`. SFT teaches what to say; preference pairs teach which of two answers is better, including what *not* to do.
- **Numbers:** a few hundred to a few thousand clean SFT examples move a narrow task (`tn13`). Quality and diversity beat volume. Pick `max_length` from the length distribution, not a guess.
- **Traps:** `ds[:2]` is a dict of lists. Truncation is silent. Unseeded shuffles aren't reproducible. `to_pandas()` loads everything into RAM. Streaming datasets have no `len()` and only buffer-shuffle.
- **Follow-ups:** "near-duplicates?" (MinHash/LSH on shingles), "PII?" (scrub with a detector before storing), "how do you version the data?" (push a dataset repo to the Hub and pin its revision, like a model).

## Next

- [05 · Fine-tune a classifier with the Trainer](05_trainer_api_fine_tune_small_model.ipynb): uses exactly this pipeline.
- [When to fine-tune and data prep](../11_finetuning/01_when_to_fine_tune_and_data_prep.ipynb) · [SFT with loss masking](../11_finetuning/02_sft_from_scratch_with_loss_masking.ipynb) · [DPO](../11_finetuning/06_dpo_preference_tuning.ipynb)
- Chunking documents for RAG: [chunking strategies](../09_rag/01_chunking_strategies_and_semantic_chunking.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html), [interview bank](../../ai_interview_prep/index.html)